In [ ]:
!pip install -U transformers

fine tune  mairan model in KDE4 dataset

# Preparing data

In [ ]:
from datasets import load_dataset
# raw_datasets = load_dataset("kde4", lang1="en", lang2="fr",
#     trust_remote_code=True)

raw_datasets = load_dataset("opus100", "en-fr")
raw_datasets

In [ ]:
# raw_datasets["train"][1]["translation"]

Processing dataset

In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
max_length = 128
def process_tokenizer(example):
  inputs = [ex["en"] for ex in example["translation"]]
  targets = [ex["fr"] for ex in example["translation"]]
  model_inputs = tokenizer(inputs,text_target =targets, max_length = max_length,truncation = True)
  return model_inputs

In [ ]:
tokenize_datasets = raw_datasets.map(
    process_tokenizer, batched = True, remove_columns = ["translation"]
)
tokenize_datasets

# Fine tuning model with Traner API

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer= tokenizer, model = model)
# test
batch = data_collator([tokenize_datasets["train"][i] for i in range(1, 3)] )
batch.keys()


matrics

In [ ]:
!pip install sacrebleu
!pip install evaluate

In [ ]:
import evaluate
metric = evaluate.load("sacrebleu")

predictions = [
    "This plugin lets you translate web pages between several languages automatically."
]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

In [ ]:
import numpy as np


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100s in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback


args = Seq2SeqTrainingArguments(
    "marian-finetuned-kde4-en-to-fr",
    eval_strategy="epoch",       # evaluate after each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenize_datasets["train"],
    eval_dataset=tokenize_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Evaluate pretrained baseline BEFORE training starts
class EvaluateFirstStepCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        control.should_evaluate = True

trainer.add_callback(EvaluateFirstStepCallback())


In [ ]:
trainer.train()

In [ ]:
trainer.add_callback(EvaluateFirstStepCallback())


In [ ]:
trainer.push_to_hub(tags="translation", commit_message="Training complete")